# Theorem 1: Fisher Metric Contraction

Verifies Theorem 1: η^(ℓ) ≤ η^(ℓ-1)(1-ε₀). The pullback g^(k) = J^T g^(k-1) J contracts.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## 1. Analytical Verification

In [ ]:
from src.proofs.theorem1_fisher_transform import verify_pushforward_numerically, run_all_verifications

results = run_all_verifications()
print("Theorem 1 verification results:")
for k, v in results.items():
    print(f"  {k}: {'PASS' if v else 'FAIL'}")


## 2. Empirical Contraction Across Depth

In [ ]:
from src.core.fisher.fisher_metric import FisherMetric
import torch

fm = FisherMetric()
n = 16
G = torch.eye(n)
eta_vals = [G.diag().max().item()]

depth = 30
for ell in range(depth):
    # Random critical-init Jacobian (sigma_w/sqrt(n) per element)
    sigma_w = 1.4
    J = torch.randn(n, n) * (sigma_w / n**0.5)
    G = fm.pullback(G, J)  # g^(ell) = J^T g^(ell-1) J
    eta_vals.append(torch.linalg.eigvalsh(G).max().item())

plt.figure(figsize=(7, 4))
plt.semilogy(range(depth+1), eta_vals, "o-", ms=4)
plt.xlabel("Layer ℓ"); plt.ylabel("max eigenvalue η^(ℓ)")
plt.title("Theorem 1: Fisher metric max eigenvalue decays with depth")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Verify contraction rate
from scipy.stats import linregress
import numpy as np
log_eta = np.log(np.array(eta_vals) + 1e-30)
slope, _, r, _, _ = linregress(range(len(log_eta)), log_eta)
print(f"Contraction rate (slope in log scale): {slope:.4f}")
print(f"R² of exponential fit: {r**2:.4f}")
